<a href="https://colab.research.google.com/github/SaiOhmSaiePaine/food_not_food_text_classifier/blob/main/Food_not_Food_Text_Classification_Customized_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Food not Food Text Classification Model

In [1]:
# Download necessary packages (note: If you run this notebook on google colab, you might have to uncomment this cell).
try:
  import datasets, evaluate, accelerate
  import gradio as gr
except ModuleNotFoundError:
  !pip install -U datasets evaluate accelerate
  import datasets, evaluate, accelerate
  import gradio as gr

In [2]:
# 1. Import necessary packages
import pprint
from pathlib import Path

import numpy as np
import torch

import datasets
import evaluate

from transformers import pipeline
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 2. Setup variables for model training and saving pipeline
DATASET_NAME = "mrdbourke/learn_hf_food_not_food_image_captions"
MODEL_NAME = "distilbert/distilbert-base-uncased"
MODEL_SAVE_DIR_NAME = "models/learn_hf_food_not_food_text_classifier-distilbert-base-uncased"

# 3. Create a directory for saving models
print(f'[INFO] Creating a directory for saving models: {MODEL_SAVE_DIR_NAME}')
model_save_dir = Path(MODEL_SAVE_DIR_NAME)
model_save_dir.mkdir(parents=True, exist_ok=True)

# 4. Load and preprocess the dataset from the Hugging Face hub
print(f'[INFO] Downloading the dataset from Hugging Face hub; name: {DATASET_NAME}')
dataset = datasets.load_dataset(path=DATASET_NAME)

# Create mappings from id2label and label2id (adjust these for your target dataset, can also create these programmatically)
unique_labels = sorted(list(set(dataset['train']['label'])), reverse=True)

id2label = {idx: label for idx, label in enumerate(unique_labels)}
label2id = {label: idx for idx, label in id2label.items()}

# id2label = {0 : 'not_food', 1 : 'food'}
# label2id = {'not_food' : 0, 'food': 1}

print(id2label)
print(label2id)

# Create function to map IDs to labels in dataset
def map_labels_to_number(example):
  example['label'] = label2id[example['label']]

  return example

# Map preprocessing function to dataset
dataset = dataset['train'].map(map_labels_to_number)

# Split the dataset into train/test set
dataset = dataset.train_test_split(test_size=0.2, seed=42)

# 5. Import a tokenizer and map it to the dataset
print(f'[INFO] Tokening text for model training with tokenizer: {MODEL_NAME}')

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=MODEL_NAME,
                                          use_fast=True)

# Create a preprocessing function to tokenize text
def tokenize_text(examples):
  return tokenizer(examples['text'],
                   padding=True,
                   truncation=True)

tokenized_dataset = dataset.map(function=tokenize_text,
                                batched=True,
                                batch_size=1000)


# 6. Set up evaluation metric & function to evaluate our model
accuracy_metric = evaluate.load('accuracy')

def compute_accuracy(predictions_and_labels):
  predictions, labels = predictions_and_labels

  if len(predictions) >= 2:
    predictions = np.argmax(predictions, axis=1)

  return accuracy_metric.compute(predictions=predictions, references=labels) # note: use "references" parameters rather than labels


# 7. Import the model and prepare it for training
print(f"[INFO] Model Name: {MODEL_NAME}")

model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path=MODEL_NAME,
    num_labels =2,
    id2label=id2label,
    label2id=label2id
)

print("[INFO] Model loading complete!")

# Setup TrainingArguments
training_args = TrainingArguments(
    output_dir = model_save_dir,
    learning_rate = 0.0001,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size = 32,
    num_train_epochs = 10,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    save_total_limit = 3,
    use_cpu = False,
    seed = 42,
    load_best_model_at_end = True,
    logging_strategy = "epoch",
    report_to = "none",
    push_to_hub = False,
    hub_private_repo = False # Note: if set to False, your model will be publically available

)

# Create Trainer instance for training the model
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_accuracy
)

# 8. Train the model on the text dataset
print('[INFO] Connecting model training...')
results = trainer.train()

# 9. Save the trained model (note: this will overwrite our previous model, this is ok)
print(f"[INFO] Model saving complete, saving model to local path: {model_save_dir}")
trainer.save_model(output_dir=model_save_dir)

# 10. Push the model to the Hugging Face hub
model_upload_url = trainer.push_to_hub(
    commit_message = "Uploading food not food text classification model"
)
print(f"[INFO] Model upload complete, model available at: {model_upload_url}")


# 11. Evaluate the model on the test data
print(f"[INFO] Performing evaluation on the test dataset...")
predictions_all = trainer.predict(tokenized_dataset['test'])
predictions_values = predictions_all.predictions
predictions_metrics = predictions_all.metrics


print("[INFO] Prediction metrics on the test data: ")
pprint.pprint(predictions_metrics)

[INFO] Creating a directory for saving models: models/learn_hf_food_not_food_text_classifier-distilbert-base-uncased
[INFO] Downloading the dataset from Hugging Face hub; name: mrdbourke/learn_hf_food_not_food_image_captions


README.md:   0%|          | 0.00/1.32k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/250 [00:00<?, ? examples/s]

{0: 'not_food', 1: 'food'}
{'not_food': 0, 'food': 1}


...

New Data Upload               : |          |  0.00B /  0.00B            

  ...uncased/model.safetensors:   2%|2         | 5.90MB /  268MB            

  ...uncased/training_args.bin:   7%|6         |   346B / 5.20kB            

[INFO] Model upload complete, model available at: https://huggingface.co/SaiePaine/learn_hf_food_not_food_text_classifier-distilbert-base-uncased/commit/584f794caa0caea36cf3d49df7d66d7ebb6c23cc
[INFO] Performing evaluation on the test dataset...


[INFO] Prediction metrics on the test data: 
{'test_accuracy': 1.0,
 'test_loss': 0.0004956427146680653,
 'test_runtime': 0.0939,
 'test_samples_per_second': 532.402,
 'test_steps_per_second': 21.296}


In [13]:
# Call the model and testing it works on a custom sample text
huggingface_model_path = 'SaiePaine/learn_hf_food_not_food_text_classifier-distilbert-base-uncased'

food_not_food_classifier = pipeline(
    task='text-classification',
    model=huggingface_model_path,
    device=torch.device("cuda") if torch.cuda.is_available() else "cpu",
    top_k=1,
    batch_size=32
)

food_not_food_classifier("Sein Linn is eating KFC.")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[[{'label': 'food', 'score': 0.9407318234443665}]]